In [3]:
import json
from PIL import Image
import torch
from torchvision import transforms

metadata_file = "/tmp/exploration_data/metadata.jsonl"
metadata = []

with open(metadata_file, "r") as f:
    for line in f:
        metadata.append(json.loads(line))

# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # or model input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

images = []
for m in metadata:
    img = Image.open(m["image_path"]).convert("RGB")
    img_tensor = transform(img)
    images.append(img_tensor)

images = torch.stack(images)  # shape: [N, 3, 224, 224]


In [8]:
images = []
for m in metadata:
    
    image_path = m['image_path'] 
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img)
    images.append(img_tensor)


In [10]:
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch

model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)

# Load and preprocess images with PIL
images = [Image.open(m["image_path"]).convert("RGB") for m in metadata]

# Only prepare pixel values (not text)
inputs = processor(images=images[0], return_tensors="pt", padding=True)

with torch.no_grad():
    embeddings = model.get_image_features(**inputs)  # shape: [N, D]


In [11]:
import faiss
import numpy as np

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)  # L2 distance
index.add(embeddings.cpu().numpy())

# Store mapping from vector ID to metadata
id_to_metadata = {i: m for i, m in enumerate(metadata)}


In [13]:
query_text = "fire_hydrant"
text_inputs = processor(text=[query_text], return_tensors="pt")
with torch.no_grad():
    query_embedding = model.get_text_features(**text_inputs)

# Search in FAISS
D, I = index.search(query_embedding.cpu().numpy(), k=1)  # top 5 results

for i in I[0]:
    print("Found at coordinates:", id_to_metadata[i]["robot_x"], id_to_metadata[i]["robot_y"])


Found at coordinates: -0.0001862175024392003 1.5808207244017054e-05


In [11]:
import json

def load_jsonl(path):
    """Load JSONL file into a list of dicts"""
    with open(path, "r") as f:
        return [json.loads(line) for line in f]

def find_objects(object_name, detected_objects, metadata):
    """Find objects by class name and attach metadata"""
    results = []
    for obj in detected_objects:
        if obj["object_class"] == object_name:
            # Find closest metadata by timestamp
            closest_meta = min(metadata, key=lambda m: abs(m["timestamp"] - obj["timestamp"]))

            results.append({
                "id": obj["id"],
                "timestamp": obj["timestamp"],
                "object_class": obj["object_class"],
                "confidence": obj["confidence"],
                # already in meters (map/world space)
                "map_coords": (obj["world_x"], obj["world_y"], obj["world_z"]),
                "pixel_coords": (obj["pixel_x"], obj["pixel_y"]),
                "bbox": obj["bbox"],
                "frame_id": obj["frame_id"],
                "image_path": closest_meta["image_path"],
                "robot_pose": (closest_meta["robot_x"], closest_meta["robot_y"], closest_meta["robot_theta"])
            })
    return results


if __name__ == "__main__":
    detected_objects = load_jsonl("/tmp/exploration_data/detected_objects.jsonl")
    metadata = load_jsonl("/tmp/exploration_data/metadata.jsonl")

    # Example: search for "box"
    search_name = "- fire_hydrant"
    found = find_objects(search_name, detected_objects, metadata)

    for item in found:
        print(f"\nObject {item['id']} ({item['object_class']})")
        print(f"  Timestamp: {item['timestamp']}")
        print(f"  Confidence: {item['confidence']}")
        print(f"  Map coords (meters): {item['map_coords']}")
        print(f"  Pixel coords: {item['pixel_coords']}")
        print(f"  Bounding box: {item['bbox']}")
        print(f"  Frame ID: {item['frame_id']} | Image: {item['image_path']}")
        print(f"  Robot pose at that time: {item['robot_pose']}")



Object obj_000020 (- fire_hydrant)
  Timestamp: 301.305
  Confidence: 0.8
  Map coords (meters): (-5.281433952139928, 5.180796781488449, 0.2947627912856288)
  Pixel coords: (100, 240)
  Bounding box: [75, 215, 125, 265]
  Frame ID: 30 | Image: /tmp/exploration_data/frame_00029.png
  Robot pose at that time: (-1.5435917681328004, 2.7655673589030467, 2.2005131851866246)
